**Reconstruction-enabled version — updated 2026-08-10**

# Compare two labeled SpikeIMU segments

This notebook is intended for a WritingRing Action-0 pipeline output such as:

`outputs/action0_rectified/low-pass/aligned-board-events`

It does four things:

1. Resolves one user's **variable-length** `segmentation/` package (not `segmentation_padded/`).
2. Finds all segments whose labels exactly match `LABEL_A` and `LABEL_B`.
3. Selects one occurrence of each label and exposes their raw `(T, 21)` SpikeIMU arrays.
4. Plots:
   - A/B acceleration in m/s² on x/y/z (`SpikeIMU[:, 15:18]`), with A blue and B black.
   - One transient-score figure for A and one for B, recomputing the score **inside each selected segment** from `SpikeIMU[:, 15:21]` and overlaying Board press/lift targets.

The transient score calls the repository implementation used by alignment:

- first difference per transient channel;
- per-channel median/MAD robust normalization;
- L2 norm across the six transient channels.

Board target channel order is the aligned-segmentation contract:
`valid_press, valid_lift, transient_press, transient_lift`.


In [ ]:
from __future__ import annotations

from collections import defaultdict
import json
import math
import os
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import signal


## Configuration

Change only this cell for the usual workflow. `A_OCCURRENCE=0` means the first exported segment whose label is `LABEL_A`; set it to 1 for the second, etc.


In [ ]:
# Path can use either Windows-style or POSIX-style separators.
DATASET_ROOT = r"outputs\action0_rectified\low-pass\aligned-board-events"
USER = "user_0"
ACTION = 0

LABEL_A = "A"
LABEL_B = "B"
A_OCCURRENCE = 0
B_OCCURRENCE = 0

# Normally leave these as None/False.
REPO_ROOT_OVERRIDE = None
SAMPLING_RATE_OVERRIDE = None
SHOW_SMOOTHED_TRANSIENT = False  # alignment verification defaults to False

# Reconstruction settings from the vendor NIMU reconstruction experiment.
RECONSTRUCTION_FREQUENCIES_HZ = (0.5, 1.0, 2.0, 4.0, 8.0)
RECONSTRUCTION_SCALE_DIVISOR = 2.5
STANDARD_GRAVITY_M_S2 = 9.80665


In [ ]:
def _candidate_repo_roots(start: Path):
    start = start.resolve()
    yield start
    yield from start.parents


def find_repo_root(override=None) -> Path:
    if override is not None:
        root = Path(str(override).replace("\\", "/")).expanduser().resolve()
        if not (root / "src" / "writingring").is_dir():
            raise FileNotFoundError(f"Not a WritingRing repository root: {root}")
        return root

    for candidate in _candidate_repo_roots(Path.cwd()):
        if (candidate / "src" / "writingring").is_dir() and (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find the WritingRing repository root from the current working directory. "
        "Set REPO_ROOT_OVERRIDE in the configuration cell."
    )


REPO_ROOT = find_repo_root(REPO_ROOT_OVERRIDE)
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

# Import the exact repository implementations used by alignment/verification.
from writingring.recording_features import compute_transient_score_array
from writingring.event_alignment import smooth_transient_score


def resolve_user_path(raw_path: str | os.PathLike[str]) -> Path:
    # Forward slashes are accepted on Windows; replacing backslashes also makes
    # the example path work when the notebook is run on Linux/macOS.
    normalized = Path(str(raw_path).replace("\\", "/")).expanduser()
    if not normalized.is_absolute():
        normalized = REPO_ROOT / normalized
    return normalized.resolve()


DATASET_ROOT_PATH = resolve_user_path(DATASET_ROOT)
print("Repository root:", REPO_ROOT)
print("Dataset root:   ", DATASET_ROOT_PATH)


In [ ]:
SPIKE_SCHEMA = "signed_wavelet_events_plus_imu_v1"
ACCEL_M_S2_SLICE = slice(15, 18)
TRANSIENT_SLICE = slice(15, 21)
BOARD_TARGET_NAMES = (
    "valid_press",
    "valid_lift",
    "transient_press",
    "transient_lift",
)


def _package_has_required_arrays(directory: Path, prefix: str) -> bool:
    required = (
        f"{prefix}_spikeIMU.npy",
        f"{prefix}_labels.npy",
        f"{prefix}_segment_offsets.npy",
        f"{prefix}_segment_lengths.npy",
        f"{prefix}_board_event_targets.npy",
        f"{prefix}_segmentation_summary.json",
    )
    return directory.is_dir() and all((directory / name).is_file() for name in required)


def resolve_segmentation_package(dataset_root: Path, user: str, action: int | str) -> tuple[Path, str]:
    action_text = str(action)
    prefix = f"{user}_action_{action_text}"
    action_dir = f"action_{action_text}"

    direct_candidates = [
        dataset_root / "segmentation" / user / action_dir,
        dataset_root / user / action_dir,  # also accept DATASET_ROOT already pointing at segmentation/
    ]
    for candidate in direct_candidates:
        if _package_has_required_arrays(candidate, prefix):
            return candidate, prefix

    # Conservative fallback: find a directory containing the exact package prefix.
    matches = []
    if dataset_root.is_dir():
        for spike_path in dataset_root.rglob(f"{prefix}_spikeIMU.npy"):
            directory = spike_path.parent
            if _package_has_required_arrays(directory, prefix):
                matches.append(directory)
    matches = sorted(set(matches))
    if len(matches) == 1:
        return matches[0], prefix
    if not matches:
        raise FileNotFoundError(
            f"Could not find aligned variable-length segmentation package for {user}, action {action_text} "
            f"under {dataset_root}. Expected .../segmentation/{user}/{action_dir}/"
        )
    raise RuntimeError(
        "Multiple matching segmentation packages found; point DATASET_ROOT more narrowly:\n" +
        "\n".join(f"  - {path}" for path in matches)
    )


PACKAGE_DIR, PREFIX = resolve_segmentation_package(DATASET_ROOT_PATH, USER, ACTION)
print("Segmentation package:", PACKAGE_DIR)
print("Package prefix:       ", PREFIX)


In [ ]:
paths = {
    "spike": PACKAGE_DIR / f"{PREFIX}_spikeIMU.npy",
    "labels": PACKAGE_DIR / f"{PREFIX}_labels.npy",
    "offsets": PACKAGE_DIR / f"{PREFIX}_segment_offsets.npy",
    "lengths": PACKAGE_DIR / f"{PREFIX}_segment_lengths.npy",
    "targets": PACKAGE_DIR / f"{PREFIX}_board_event_targets.npy",
    "summary": PACKAGE_DIR / f"{PREFIX}_segmentation_summary.json",
    "segments_csv": PACKAGE_DIR / f"{PREFIX}_segments.csv",
}

spike_imu = np.load(paths["spike"], allow_pickle=False)
labels = np.load(paths["labels"], allow_pickle=False).astype(str)
segment_offsets = np.load(paths["offsets"], allow_pickle=False).astype(np.int64)
segment_lengths = np.load(paths["lengths"], allow_pickle=False).astype(np.int64)
board_event_targets = np.load(paths["targets"], allow_pickle=False)
summary = json.loads(paths["summary"].read_text(encoding="utf-8"))

if spike_imu.ndim != 2 or spike_imu.shape[1] != 21:
    raise ValueError(f"Expected SpikeIMU shape (N, 21), got {spike_imu.shape}")
if labels.ndim != 1:
    raise ValueError(f"Expected labels shape (S,), got {labels.shape}")
if segment_offsets.ndim != 1 or len(segment_offsets) != len(labels) + 1:
    raise ValueError("segment_offsets must have length segment_count + 1")
if segment_lengths.shape != labels.shape:
    raise ValueError("segment_lengths must have one value per label")
if segment_offsets[0] != 0 or segment_offsets[-1] != len(spike_imu):
    raise ValueError("segment_offsets do not span the full SpikeIMU aggregate")
if np.any(np.diff(segment_offsets) < 0):
    raise ValueError("segment_offsets must be nondecreasing")
if not np.array_equal(np.diff(segment_offsets), segment_lengths):
    raise ValueError("segment_lengths does not equal diff(segment_offsets)")
if board_event_targets.shape != (len(spike_imu), 4):
    raise ValueError(
        "Aligned-Board target array must be row-aligned with SpikeIMU and have shape (N, 4); "
        f"got {board_event_targets.shape}"
    )
if board_event_targets.dtype != np.bool_:
    # The public contract is boolean; do not silently reinterpret arbitrary numeric targets.
    raise ValueError(f"board_event_targets must be boolean, got {board_event_targets.dtype}")

if summary.get("input_kind") not in (None, "spike-imu"):
    raise ValueError(f"Expected input_kind='spike-imu', got {summary.get('input_kind')!r}")
if summary.get("feature_schema") not in (None, SPIKE_SCHEMA):
    raise ValueError(f"Expected feature_schema={SPIKE_SCHEMA!r}, got {summary.get('feature_schema')!r}")

if SAMPLING_RATE_OVERRIDE is not None:
    sampling_rate_hz = float(SAMPLING_RATE_OVERRIDE)
else:
    sampling_rate_hz = float(summary.get("sampling_rate_hz", float("nan")))
if not math.isfinite(sampling_rate_hz) or sampling_rate_hz <= 0:
    raise ValueError(
        "No valid sampling_rate_hz was found in the segmentation summary. "
        "Set SAMPLING_RATE_OVERRIDE explicitly if this is an intentional legacy artifact."
    )

print(f"SpikeIMU aggregate: {spike_imu.shape}")
print(f"Segments:           {len(labels)}")
print(f"Sampling rate:      {sampling_rate_hz:g} Hz")
print(f"Board targets:      {board_event_targets.shape}, dtype={board_event_targets.dtype}")


## Find all matching A/B segments

The table below is the selection catalog. `label_occurrence` is zero-based within each exact label. Event counts come from the already-published row-aligned Board target array; the transient score itself is **not** read from disk and will be recomputed later.


In [ ]:
def segment_bounds(segment_index: int) -> tuple[int, int]:
    if segment_index < 0 or segment_index >= len(labels):
        raise IndexError(f"segment_index out of range: {segment_index}")
    return int(segment_offsets[segment_index]), int(segment_offsets[segment_index + 1])


def segment_arrays(segment_index: int) -> tuple[np.ndarray, np.ndarray]:
    start, stop = segment_bounds(segment_index)
    return spike_imu[start:stop], board_event_targets[start:stop]


occurrence_counter = defaultdict(int)
rows = []
for segment_index, label in enumerate(labels.tolist()):
    start, stop = segment_bounds(segment_index)
    local_targets = board_event_targets[start:stop]
    occurrence = occurrence_counter[label]
    occurrence_counter[label] += 1
    rows.append(
        {
            "segment_index": segment_index,
            "label": label,
            "label_occurrence": occurrence,
            "start_row": start,
            "stop_row_exclusive": stop,
            "length_samples": stop - start,
            "sequence_duration_s": (stop - start) / sampling_rate_hz,
            "valid_press_count": int(local_targets[:, 0].sum()),
            "valid_lift_count": int(local_targets[:, 1].sum()),
            "transient_press_count": int(local_targets[:, 2].sum()),
            "transient_lift_count": int(local_targets[:, 3].sum()),
        }
    )

catalog = pd.DataFrame(rows)
match_catalog = catalog[catalog["label"].isin([LABEL_A, LABEL_B])].copy()

missing = [label for label in (LABEL_A, LABEL_B) if not (catalog["label"] == label).any()]
if missing:
    available = catalog.groupby("label").size().sort_values(ascending=False).rename("count")
    display(available.to_frame())
    raise ValueError(
        f"Exact label(s) not found for {USER}: {missing}. "
        "Labels are case-sensitive; available label counts are shown above."
    )

display(match_catalog.reset_index(drop=True))


## Select one A segment and one B segment

Change `A_OCCURRENCE` / `B_OCCURRENCE` in the configuration cell, then rerun from the catalog cell onward.


In [ ]:
def select_label_occurrence(label: str, occurrence: int) -> int:
    candidates = catalog.loc[catalog["label"] == label].sort_values("label_occurrence")
    if occurrence < 0 or occurrence >= len(candidates):
        raise IndexError(
            f"{label!r} has {len(candidates)} exported segment(s); "
            f"requested occurrence {occurrence}."
        )
    return int(candidates.iloc[occurrence]["segment_index"])


A_segment_index = select_label_occurrence(LABEL_A, A_OCCURRENCE)
B_segment_index = select_label_occurrence(LABEL_B, B_OCCURRENCE)

A_segment_spikeIMU, A_segment_board_targets = segment_arrays(A_segment_index)
B_segment_spikeIMU, B_segment_board_targets = segment_arrays(B_segment_index)

selected = catalog.loc[catalog["segment_index"].isin([A_segment_index, B_segment_index])].copy()
display(selected)

print(f"A_segment_spikeIMU: {A_segment_spikeIMU.shape}  (global segment index {A_segment_index})")
print(f"B_segment_spikeIMU: {B_segment_spikeIMU.shape}  (global segment index {B_segment_index})")


## Plot A vs B acceleration (m/s²)

SpikeIMU channels `15:18` are acceleration x/y/z in m/s². Each segment uses its own local elapsed-time axis starting at zero.


In [ ]:
def local_time_s(segment: np.ndarray) -> np.ndarray:
    return np.arange(len(segment), dtype=np.float64) / sampling_rate_hz


def plot_accel_comparison(
    segment_a: np.ndarray,
    segment_b: np.ndarray,
    *,
    label_a: str,
    label_b: str,
    segment_index_a: int,
    segment_index_b: int,
):
    time_a = local_time_s(segment_a)
    time_b = local_time_s(segment_b)
    axis_names = ("x", "y", "z")

    fig, axes = plt.subplots(3, 1, figsize=(12, 8.5), sharex=True, layout="constrained")
    for axis_i, ax in enumerate(np.asarray(axes).reshape(-1)):
        channel = 15 + axis_i
        ax.plot(time_a, segment_a[:, channel], color="blue", linewidth=1.0, label=f"{label_a} (segment {segment_index_a})")
        ax.plot(time_b, segment_b[:, channel], color="black", linewidth=1.0, label=f"{label_b} (segment {segment_index_b})")
        ax.set_ylabel(f"accel {axis_names[axis_i]}\n(m/s²)")
        ax.grid(True, alpha=0.25)
        ax.legend(loc="upper right")
    axes[-1].set_xlabel("Local elapsed time (s)")
    fig.suptitle(
        f"Acceleration comparison: {label_a} vs {label_b}\n"
        f"{USER}, action {ACTION}; {sampling_rate_hz:g} Hz"
    )
    plt.show()


plot_accel_comparison(
    A_segment_spikeIMU,
    B_segment_spikeIMU,
    label_a=LABEL_A,
    label_b=LABEL_B,
    segment_index_a=A_segment_index,
    segment_index_b=B_segment_index,
)


## Recompute transient score inside each selected segment

This is deliberately a **segment-local recomputation**. In other words, the first differences, median, and MAD are calculated separately for A and B after slicing their SpikeIMU segments. It does not slice a transient score that was computed on the full recording.

The score uses `SpikeIMU[:, 15:21]` (acceleration m/s² + gyro rad/s), exactly matching the SpikeIMU transient-channel contract used by alignment.

For the event overlay:

- valid press: red solid line;
- valid lift: orange dashed line;
- transient press/lift: lighter dotted lines.

The first two styles follow the alignment verification plot. The transient event styles are added here because aligned segmentation publishes separate transient target channels.


In [ ]:
def recompute_segment_transient_score(segment: np.ndarray) -> np.ndarray:
    if segment.ndim != 2 or segment.shape[1] != 21 or len(segment) == 0:
        raise ValueError(f"Expected a nonempty (T, 21) SpikeIMU segment, got {segment.shape}")
    # Important: calculate on this segment only, as requested.
    return compute_transient_score_array(segment[:, TRANSIENT_SLICE])


def _verification_like_ymax(score: np.ndarray) -> float:
    ymax = float(np.quantile(score, 0.995) * 1.1)
    if not math.isfinite(ymax) or ymax <= 0.0:
        ymax = max(float(np.max(score)), 1.0)
    return ymax


def _draw_target_lines(ax, targets: np.ndarray, time_s: np.ndarray) -> None:
    styles = (
        (0, "tab:red", "-", 0.90, 1.5, "Board valid press"),
        (1, "tab:orange", "--", 0.90, 1.5, "Board valid lift"),
        (2, "tab:red", ":", 0.40, 0.9, "Board transient press"),
        (3, "tab:orange", ":", 0.40, 0.9, "Board transient lift"),
    )
    for channel, color, linestyle, alpha, linewidth, label in styles:
        indices = np.flatnonzero(targets[:, channel])
        for j, idx in enumerate(indices):
            ax.axvline(
                float(time_s[idx]),
                color=color,
                linestyle=linestyle,
                alpha=alpha,
                linewidth=linewidth,
                label=label if j == 0 else None,
            )


def plot_segment_transient(
    segment: np.ndarray,
    targets: np.ndarray,
    *,
    label: str,
    segment_index: int,
    show_smoothed: bool = False,
):
    if targets.shape != (len(segment), 4):
        raise ValueError(f"Expected row-aligned target shape ({len(segment)}, 4), got {targets.shape}")

    score = recompute_segment_transient_score(segment)
    time_s = local_time_s(segment)

    fig, ax = plt.subplots(figsize=(12, 4.2), layout="constrained")
    ax.plot(time_s, score, color="tab:blue", linewidth=0.8, label="Ring transient score")

    if show_smoothed:
        smooth = smooth_transient_score(score, window_samples=5)
        ax.plot(time_s, smooth, color="tab:cyan", linewidth=0.55, label="Smoothed transient (5 samples)")

    _draw_target_lines(ax, targets, time_s)

    ax.set_xlim(0.0, max(float(time_s[-1]) if len(time_s) > 1 else 0.0, 1.0 / sampling_rate_hz))
    ax.set_ylim(0.0, _verification_like_ymax(score))
    ax.set_xlabel("Local elapsed time (s)")
    ax.set_ylabel("Transient score")
    ax.set_title(
        f"{label} transient score — segment {segment_index}\n"
        f"recomputed from this segment's SpikeIMU[:, 15:21]"
    )
    ax.grid(True, alpha=0.25)

    handles, legend_labels = ax.get_legend_handles_labels()
    unique = {}
    for handle, legend_label in zip(handles, legend_labels):
        unique.setdefault(legend_label, handle)
    ax.legend(unique.values(), unique.keys(), loc="upper right", fontsize=8)
    plt.show()

    return score


In [ ]:
A_transient_score = plot_segment_transient(
    A_segment_spikeIMU,
    A_segment_board_targets,
    label=LABEL_A,
    segment_index=A_segment_index,
    show_smoothed=SHOW_SMOOTHED_TRANSIENT,
)


In [ ]:
B_transient_score = plot_segment_transient(
    B_segment_spikeIMU,
    B_segment_board_targets,
    label=LABEL_B,
    segment_index=B_segment_index,
    show_smoothed=SHOW_SMOOTHED_TRANSIENT,
)


## Reconstruct acceleration from the 15 Custom Wavelet event channels

This section follows the vendor NIMU reconstruction experiment, adapted to the **current occurrence-aligned SpikeIMU artifact**.

- SpikeIMU channels `0:15` are interpreted in axis-major / frequency-minor order: 5 bands for x, then 5 for y, then 5 for z.
- The band frequencies are `[0.5, 1, 2, 4, 8] Hz`.
- Each event channel is convolved with the time-reversed `accelerationWavelet`; the five reconstructed bands are summed and divided by `2.5`.
- The reconstructed acceleration is treated as g-domain acceleration and multiplied by `9.80665` to compare against SpikeIMU channels `15:18` in m/s².
- Reconstruction is performed **inside each selected segment**, so samples outside that segment contribute zero context.

The old vendor script reconstructed detection-aligned events and included the max-filter confirmation delay in its kernel placement. Current WritingRing SpikeIMU events have already been shifted back to their extrema occurrence rows, so this notebook keeps the vendor's band-dependent `si/2` wavelet-delay compensation but does **not** apply the max-filter confirmation delay a second time.

If the selected dataset is an `AbsRectify` publication, these 15 channels have already lost event polarity. Reconstruction therefore uses the published rectified amplitudes as-is; it cannot recover signs that are no longer present in the artifact.


In [ ]:
RECON_FREQS = np.asarray(RECONSTRUCTION_FREQUENCIES_HZ, dtype=np.float64)
EVENT_CHANNEL_COUNT = 15
EVENTS_PER_AXIS = 5
AXIS_NAMES = ("x", "y", "z")


def acceleration_wavelet_numpy(M: int, s: float) -> np.ndarray:
    """NumPy equivalent of vendor/Neuromorphic-Gravity/python-pipeline/wavelets.py."""
    M = int(M)
    s = float(s)
    if M <= 0 or not math.isfinite(s) or s <= 0:
        raise ValueError(f"Invalid accelerationWavelet parameters: M={M}, s={s}")
    x = (np.arange(M, dtype=np.float64) - (M - 1) / 2.0) / s
    support = (x > -0.5) & (x < 0.5)
    wavelet = support * (29.0 / 4.0) * x * (4.0 * x**2 - 1.0)
    return np.sqrt(1.0 / s) * wavelet


def _reconstruction_kernel(
    *,
    sampling_rate_hz: float,
    frequency_hz: float,
    max_kernel_length: int,
) -> np.ndarray:
    """Build the vendor-style band kernel for occurrence-aligned events.

    The vendor code places the impulse at maxM//2 - si//2 to compensate the
    wavelet/IIR band delay. It also enlarged the kernel by delayMaxFilter to
    compensate causal extrema confirmation latency. Current SpikeIMU events
    are already occurrence-aligned, so only the si//2 placement is retained.
    """
    si_float = float(sampling_rate_hz) / float(frequency_hz)
    si = int(round(si_float))
    if not np.isclose(si_float, si, rtol=0.0, atol=1e-12):
        raise ValueError(
            f"Vendor reconstruction expects an integer wavelet width; "
            f"sampling_rate/frequency = {si_float} for {frequency_hz:g} Hz"
        )

    impulse_index = max_kernel_length // 2 - si // 2
    if impulse_index < 0 or impulse_index >= max_kernel_length:
        raise ValueError(
            f"Kernel is too short for {frequency_hz:g} Hz: "
            f"length={max_kernel_length}, width={si}"
        )

    impulse = signal.unit_impulse(max_kernel_length, impulse_index)
    wavelet = acceleration_wavelet_numpy(si, si)[::-1]
    return signal.convolve(impulse, wavelet, mode="same")


def reconstruct_acceleration_from_spike_events(
    segment: np.ndarray,
    *,
    sampling_rate_hz: float,
    frequencies_hz: np.ndarray = RECON_FREQS,
    scale_divisor: float = RECONSTRUCTION_SCALE_DIVISOR,
) -> tuple[np.ndarray, np.ndarray]:
    """Reconstruct x/y/z acceleration from SpikeIMU[:, 0:15].

    Returns `(reconstructed_g, reconstructed_m_s2)`, each with shape `(T, 3)`.
    """
    values = np.asarray(segment, dtype=np.float64)
    if values.ndim != 2 or values.shape[1] != 21 or len(values) == 0:
        raise ValueError(f"Expected a nonempty (T, 21) SpikeIMU segment, got {values.shape}")
    if not np.isfinite(values).all():
        raise ValueError("SpikeIMU segment must contain only finite values")

    frequencies = np.asarray(frequencies_hz, dtype=np.float64)
    if frequencies.shape != (EVENTS_PER_AXIS,) or not np.isfinite(frequencies).all() or np.any(frequencies <= 0):
        raise ValueError(f"Expected exactly five positive reconstruction frequencies, got {frequencies}")
    if not math.isfinite(float(sampling_rate_hz)) or float(sampling_rate_hz) <= 0:
        raise ValueError("sampling_rate_hz must be positive and finite")
    if not math.isfinite(float(scale_divisor)) or float(scale_divisor) == 0:
        raise ValueError("scale_divisor must be finite and nonzero")

    widths = np.asarray([int(round(float(sampling_rate_hz) / f)) for f in frequencies], dtype=int)
    # The vendor experiment uses maxM = 2 * max(width). We keep that full
    # kernel even for short label segments so the low-frequency band remains valid.
    max_kernel_length = int(2 * widths.max())

    events = values[:, :EVENT_CHANNEL_COUNT].reshape(len(values), 3, EVENTS_PER_AXIS)
    reconstructed_g = np.zeros((len(values), 3), dtype=np.float64)

    kernels = [
        _reconstruction_kernel(
            sampling_rate_hz=float(sampling_rate_hz),
            frequency_hz=float(frequency),
            max_kernel_length=max_kernel_length,
        )
        for frequency in frequencies
    ]

    for axis_index in range(3):
        for band_index, kernel in enumerate(kernels):
            reconstructed_g[:, axis_index] += signal.convolve(
                events[:, axis_index, band_index],
                kernel,
                mode="same",
            ) / float(scale_divisor)

    reconstructed_m_s2 = reconstructed_g * float(STANDARD_GRAVITY_M_S2)
    return reconstructed_g, reconstructed_m_s2


A_reconstructed_accel_g, A_reconstructed_accel_m_s2 = reconstruct_acceleration_from_spike_events(
    A_segment_spikeIMU,
    sampling_rate_hz=sampling_rate_hz,
)
B_reconstructed_accel_g, B_reconstructed_accel_m_s2 = reconstruct_acceleration_from_spike_events(
    B_segment_spikeIMU,
    sampling_rate_hz=sampling_rate_hz,
)

print("A reconstructed accel:", A_reconstructed_accel_m_s2.shape, "m/s²")
print("B reconstructed accel:", B_reconstructed_accel_m_s2.shape, "m/s²")


### Original vs reconstructed acceleration — 3 × 2 comparison

Rows are x/y/z. The left column is the selected A segment and the right column is the selected B segment. Each panel overlays the original SpikeIMU acceleration in m/s² with the event-based reconstruction in m/s².


In [ ]:
def plot_original_vs_reconstructed_acceleration(
    segment_a: np.ndarray,
    reconstructed_a_m_s2: np.ndarray,
    segment_b: np.ndarray,
    reconstructed_b_m_s2: np.ndarray,
):
    segments = (segment_a, segment_b)
    reconstructions = (reconstructed_a_m_s2, reconstructed_b_m_s2)
    labels_local = (LABEL_A, LABEL_B)
    segment_indices = (A_segment_index, B_segment_index)

    fig, axes = plt.subplots(3, 2, figsize=(15, 10), sharex="col", layout="constrained")
    for column, (segment, reconstructed, label, segment_index) in enumerate(
        zip(segments, reconstructions, labels_local, segment_indices, strict=True)
    ):
        if reconstructed.shape != (len(segment), 3):
            raise ValueError(
                f"Reconstruction for {label!r} must have shape ({len(segment)}, 3), "
                f"got {reconstructed.shape}"
            )
        time_s = local_time_s(segment)
        original_accel = segment[:, ACCEL_M_S2_SLICE]

        for axis_index, axis_name in enumerate(AXIS_NAMES):
            ax = axes[axis_index, column]
            ax.plot(
                time_s,
                original_accel[:, axis_index],
                color="black",
                linewidth=1.0,
                label="Original accel",
            )
            ax.plot(
                time_s,
                reconstructed[:, axis_index],
                color="tab:red",
                linewidth=1.0,
                alpha=0.9,
                label="Reconstructed from events",
            )
            ax.set_ylabel(f"accel {axis_name} (m/s²)")
            ax.grid(True, alpha=0.25)
            if axis_index == 0:
                ax.set_title(f"{label} — segment {segment_index}")
                ax.legend(loc="upper right", fontsize=8)
        axes[-1, column].set_xlabel("Local elapsed time (s)")

    fig.suptitle(
        "Original vs Custom Wavelet event reconstruction\n"
        f"{USER}, action {ACTION}; reconstruction × {STANDARD_GRAVITY_M_S2:g} → m/s²"
    )
    plt.show()


plot_original_vs_reconstructed_acceleration(
    A_segment_spikeIMU,
    A_reconstructed_accel_m_s2,
    B_segment_spikeIMU,
    B_reconstructed_accel_m_s2,
)


### Original transient score vs reconstruction-based transient score

The original score is the existing segment-local score from SpikeIMU channels `15:21` (`accel_m/s² + gyro_rad/s`). For the reconstruction score, only the acceleration triplet is replaced by reconstructed acceleration in m/s²; the segment's original gyro channels `18:21` are retained. The same robust first-difference median/MAD normalization and L2 norm are then recomputed from scratch inside that segment.


In [ ]:
def recompute_transient_score_with_reconstructed_accel(
    segment: np.ndarray,
    reconstructed_accel_m_s2: np.ndarray,
) -> np.ndarray:
    if reconstructed_accel_m_s2.shape != (len(segment), 3):
        raise ValueError(
            f"Expected reconstructed accel shape ({len(segment)}, 3), "
            f"got {reconstructed_accel_m_s2.shape}"
        )
    reconstructed_transient_channels = np.column_stack(
        (reconstructed_accel_m_s2, segment[:, 18:21])
    )
    return compute_transient_score_array(reconstructed_transient_channels)


A_reconstructed_transient_score = recompute_transient_score_with_reconstructed_accel(
    A_segment_spikeIMU,
    A_reconstructed_accel_m_s2,
)
B_reconstructed_transient_score = recompute_transient_score_with_reconstructed_accel(
    B_segment_spikeIMU,
    B_reconstructed_accel_m_s2,
)


def plot_transient_score_reconstruction_comparison():
    cases = (
        (
            LABEL_A,
            A_segment_index,
            A_segment_spikeIMU,
            A_segment_board_targets,
            A_transient_score,
            A_reconstructed_transient_score,
        ),
        (
            LABEL_B,
            B_segment_index,
            B_segment_spikeIMU,
            B_segment_board_targets,
            B_transient_score,
            B_reconstructed_transient_score,
        ),
    )

    fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), layout="constrained")
    for ax, (label, segment_index, segment, targets, original_score, reconstructed_score) in zip(
        np.asarray(axes).reshape(-1), cases, strict=True
    ):
        time_s = local_time_s(segment)
        ax.plot(
            time_s,
            original_score,
            color="black",
            linewidth=0.9,
            label="Original transient score",
        )
        ax.plot(
            time_s,
            reconstructed_score,
            color="tab:red",
            linewidth=0.9,
            alpha=0.9,
            label="Reconstruction transient score",
        )
        _draw_target_lines(ax, targets, time_s)

        combined_score = np.concatenate((original_score, reconstructed_score))
        ax.set_ylim(0.0, _verification_like_ymax(combined_score))
        ax.set_xlim(
            0.0,
            max(float(time_s[-1]) if len(time_s) > 1 else 0.0, 1.0 / sampling_rate_hz),
        )
        ax.set_xlabel("Local elapsed time (s)")
        ax.set_ylabel("Transient score")
        ax.set_title(f"{label} — segment {segment_index}")
        ax.grid(True, alpha=0.25)

        handles, legend_labels = ax.get_legend_handles_labels()
        unique = {}
        for handle, legend_label in zip(handles, legend_labels):
            unique.setdefault(legend_label, handle)
        ax.legend(unique.values(), unique.keys(), loc="upper right", fontsize=7)

    fig.suptitle(
        "Segment-local transient score: original vs reconstructed acceleration\n"
        "Reconstruction score uses reconstructed accel (m/s²) + original gyro (rad/s)"
    )
    plt.show()


plot_transient_score_reconstruction_comparison()


## Useful variables after running

- `match_catalog`: every exported A/B segment found for this user/action.
- `A_segment_spikeIMU`, `B_segment_spikeIMU`: selected raw variable-length `(T, 21)` arrays.
- `A_segment_board_targets`, `B_segment_board_targets`: row-aligned `(T, 4)` boolean Board targets.
- `A_transient_score`, `B_transient_score`: segment-local recomputed transient scores.
- `A_segment_index`, `B_segment_index`: global exported segment indices within the user/action package.

- `A_reconstructed_accel_g`, `B_reconstructed_accel_g`: reconstructed acceleration before unit conversion.
- `A_reconstructed_accel_m_s2`, `B_reconstructed_accel_m_s2`: reconstruction multiplied by `9.80665`.
- `A_reconstructed_transient_score`, `B_reconstructed_transient_score`: segment-local transient scores after replacing original acceleration with the reconstruction while retaining original gyro.
